# Обучение Qwen3.5-4B LoRA-адаптеров

Этот notebook содержит исходный QLoRA training run. В финальном решении из этого run используется адаптер категории **«БАД»**.

Для двух категорий обучение запускалось параллельно на двух T4. Позднее адаптер категории «Легковоспламеняющиеся» был заменён отдельной улучшенной версией.


In [1]:
# 0. Dependencies
!pip install -q -U "transformers==5.14.0" peft accelerate bitsandbytes safetensors huggingface_hub
!pip cache purge >/dev/null 2>&1 || true
!pip install -q causal-conv1d flash-linear-attention || true
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 91.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 101.5 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
  Installing build dependencies ... done
  Getting requirements to bui

In [2]:
# 1. Config
from pathlib import Path
import os, gc, json, math, random, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
from huggingface_hub import snapshot_download
from IPython.display import FileLink, display

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
VAL_N = 600
MODEL_ID = "Qwen/Qwen3.5-4B"

CONTACT_SHEET_SIZE = 576
MAX_IMAGES = 5
JPEG_QUALITY = 88

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LR = 1e-4
EPOCHS = 1
GRAD_ACCUM = 8
MAX_DESCRIPTION_CHARS = 2200
BALANCED_SAMPLING = True

ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working/ecup_phase3")
HF_ROOT = Path("/kaggle/working/hf_cache")
SHEETS_DIR = WORK/"contact_sheets"
ADAPTERS_DIR = WORK/"adapters"

for p in [WORK, HF_ROOT, SHEETS_DIR, ADAPTERS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_ROOT/"hub")
os.environ["HF_XET_CACHE"] = str(HF_ROOT/"xet")

assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, "Выбери Kaggle T4 x2"

for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(i, torch.cuda.get_device_name(i), round(p.total_memory/2**30,2), "GB")

def disk_status(title):
    total, used, free = shutil.disk_usage("/kaggle/working")
    print(f"{title}: used={used/2**30:.2f} GB free={free/2**30:.2f} GB")

disk_status("Start")

0 Tesla T4 14.56 GB
1 Tesla T4 14.56 GB
Start: used=0.00 GB free=19.50 GB


In [3]:
# 2. Locate dataset / images
def find_data_csv(root):
    files = list(root.rglob("data.csv"))
    if not files:
        raise FileNotFoundError("data.csv not found")
    return files[0]

def find_images_root(root):
    candidates = []
    for p in root.rglob("images"):
        if not p.is_dir():
            continue
        dirs = [x for x in list(p.iterdir())[:100] if x.is_dir()]
        if dirs:
            score = sum(x.name.isdigit() for x in dirs) / len(dirs)
            if score >= 0.7:
                candidates.append((score,p))
    if not candidates:
        raise FileNotFoundError("images/<id> not found")
    return max(candidates, key=lambda x:x[0])[1]

DATA_CSV = find_data_csv(ROOT)
IMAGES_ROOT = find_images_root(ROOT)
df = pd.read_csv(DATA_CSV)

IMG_EXTS = {".jpg",".jpeg",".png",".webp"}

def id_str(x):
    if isinstance(x,(float,np.floating)) and float(x).is_integer():
        return str(int(x))
    return str(x)

def sort_key(p):
    try: return (0,int(p.stem))
    except: return (1,p.name)

def get_images(pid):
    folder = IMAGES_ROOT/id_str(pid)
    if not folder.exists():
        return []
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS],
        key=sort_key
    )[:MAX_IMAGES]

df["image_paths"] = [get_images(x) for x in tqdm(df["id"], desc="Link images")]
df["n_images"] = df["image_paths"].map(len)

print("Rows:", len(df))
display(pd.crosstab(df["category"], df["label"], margins=True))

Link images:   0%|          | 0/12971 [00:00<?, ?it/s]

Rows: 12971


label,0,1,All
category,,,
БАД,1905,5564,7469
Легковоспламеняющиеся,5304,198,5502
All,7209,5762,12971


In [4]:
# 3. Fixed validation (reuse Phase 2 if provided)
validation_files = list(ROOT.rglob("validation_ids.csv"))

if validation_files:
    fixed = pd.read_csv(validation_files[0])
    ids = set(fixed["id"].astype(str))
    mask = df["id"].astype(str).isin(ids)
    val_df = df[mask].copy().reset_index(drop=True)
    train_df = df[~mask].copy().reset_index(drop=True)
    assert len(val_df) == len(fixed)
    print("Reused:", validation_files[0])
else:
    strata = df["category"].astype(str) + "__" + df["label"].astype(str)
    train_df, val_df = train_test_split(
        df, test_size=VAL_N, random_state=SEED, stratify=strata
    )
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    print("Recreated deterministic SEED=42 validation")

val_df[["id","category","label","n_images"]].to_csv(
    WORK/"validation_ids.csv", index=False
)

print("Train:", len(train_df), "Val:", len(val_df))
display(pd.crosstab(val_df["category"], val_df["label"], margins=True))

Recreated deterministic SEED=42 validation
Train: 12371 Val: 600


label,0,1,All
category,,,
БАД,88,258,346
Легковоспламеняющиеся,245,9,254
All,333,267,600


In [5]:
# 4. Contact sheets for all rows
def fit_tile(img, box):
    img = img.convert("RGB")
    img.thumbnail(box, Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", box, "white")
    canvas.paste(img, ((box[0]-img.width)//2, (box[1]-img.height)//2))
    return canvas

def make_sheet(paths, out_path):
    if out_path.exists():
        return str(out_path)
    paths = list(paths)[:MAX_IMAGES]
    if not paths:
        Image.new("RGB",(CONTACT_SHEET_SIZE,CONTACT_SHEET_SIZE),"white").save(
            out_path, "JPEG", quality=JPEG_QUALITY
        )
        return str(out_path)

    n = len(paths)
    if n == 1: cols, rows = 1,1
    elif n <= 4: cols, rows = 2, math.ceil(n/2)
    else: cols, rows = 3,2

    gap = 4
    tw = (CONTACT_SHEET_SIZE-gap*(cols-1))//cols
    th = (CONTACT_SHEET_SIZE-gap*(rows-1))//rows
    sheet = Image.new("RGB",(CONTACT_SHEET_SIZE,CONTACT_SHEET_SIZE),"white")

    for i,p in enumerate(paths):
        try:
            with Image.open(p) as im:
                tile = fit_tile(im,(tw,th))
        except:
            tile = Image.new("RGB",(tw,th),"white")
        sheet.paste(tile, ((i%cols)*(tw+gap),(i//cols)*(th+gap)))

    sheet.save(out_path,"JPEG",quality=JPEG_QUALITY,optimize=False)
    return str(out_path)

all_df = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val")
], ignore_index=True)

def work_sheet(row):
    p = SHEETS_DIR/f"{id_str(row.id)}.jpg"
    return row.Index, make_sheet(row.image_paths,p)

paths = [None]*len(all_df)
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = [ex.submit(work_sheet,row) for row in all_df.itertuples()]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Contact sheets"):
        i,p = f.result()
        paths[i] = p

all_df["sheet_path"] = paths
MANIFEST = WORK/"train_manifest.csv"
all_df[[
    "id","name","description","category","label","n_images","split","sheet_path"
]].to_csv(MANIFEST,index=False)

display(all_df.groupby(["split","category","label"]).size().rename("n").reset_index())
disk_status("After sheets")

Contact sheets:   0%|          | 0/12971 [00:00<?, ?it/s]

,split,category,label,n
0,train,БАД,0,1817
1,train,БАД,1,5306
2,train,Легковоспламеняющиеся,0,5059
3,train,Легковоспламеняющиеся,1,189
4,val,БАД,0,88
5,val,БАД,1,258
6,val,Легковоспламеняющиеся,0,245
7,val,Легковоспламеняющиеся,1,9


After sheets: used=0.78 GB free=18.72 GB


In [6]:
# 5. Download Qwen once
MODEL_PATH = snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=str(HF_ROOT/"hub"),
    max_workers=4,
)

xet = HF_ROOT/"xet"
if xet.exists():
    shutil.rmtree(xet, ignore_errors=True)
    xet.mkdir(parents=True, exist_ok=True)

print("MODEL_PATH:", MODEL_PATH)
disk_status("After model")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

MODEL_PATH: /kaggle/working/hf_cache/hub/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
After model: used=9.48 GB free=10.02 GB


In [7]:
# 6. Write trainer script
TRAIN_SCRIPT = WORK/"train_qwen_qlora.py"
TRAIN_SCRIPT.write_text('\nimport argparse, gc, json, math, random, time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom sklearn.metrics import f1_score, accuracy_score, confusion_matrix\n\nimport torch\nfrom torch.nn.utils import clip_grad_norm_\nfrom transformers import (\n    AutoProcessor,\n    BitsAndBytesConfig,\n    Qwen3_5ForConditionalGeneration,\n    get_cosine_schedule_with_warmup,\n)\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n\nBAD_RULES = """Правила БАД:\n- относится, если есть прямое указание БАД / биологически активная добавка / dietary supplement;\n- спортивное питание при прямом указании на спортпит не относится;\n- если явно сказано, что товар не является БАД, он не относится;\n- без маркировки БАД / dietary supplement товар не относится."""\n\nFIRE_RULES = """Правила Легковоспламеняющиеся:\n- относится: самостоятельный источник воспламенения; содержит горючее вещество/ЛВЖ/горючий газ; опасный товар входит в комплект;\n- не относится: устройство лишь используется с огнем/топливом, но не содержит его;\n- не относится: горючее содержимое отсутствует в поставке;\n- не относится: источник воспламенения встроен;\n- не относится: горючий материал только компонент;\n- не относится: опасный предмет не входит в комплект."""\n\ndef args_parser():\n    p = argparse.ArgumentParser()\n    p.add_argument("--model_path", required=True)\n    p.add_argument("--manifest", required=True)\n    p.add_argument("--category", required=True)\n    p.add_argument("--output_dir", required=True)\n    p.add_argument("--r", type=int, default=32)\n    p.add_argument("--alpha", type=int, default=64)\n    p.add_argument("--dropout", type=float, default=0.05)\n    p.add_argument("--lr", type=float, default=1e-4)\n    p.add_argument("--epochs", type=int, default=1)\n    p.add_argument("--grad_accum", type=int, default=8)\n    p.add_argument("--weight_decay", type=float, default=0.01)\n    p.add_argument("--warmup_ratio", type=float, default=0.05)\n    p.add_argument("--max_description_chars", type=int, default=2200)\n    p.add_argument("--balanced_sampling", type=int, default=1)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--log_every", type=int, default=25)\n    p.add_argument("--smoke_only", action="store_true")\n    return p.parse_args()\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\ndef trim_desc(value, max_chars):\n    s = "" if pd.isna(value) else str(value)\n    if len(s) <= max_chars:\n        return s\n    h = int(max_chars * 0.70)\n    return s[:h] + "\\n...[середина сокращена]...\\n" + s[-(max_chars-h):]\n\ndef build_prompt(row, max_chars):\n    cat = str(row["category"])\n    rules = BAD_RULES if cat == "БАД" else FIRE_RULES\n    name = "" if pd.isna(row["name"]) else str(row["name"])\n    desc = trim_desc(row["description"], max_chars)\n    return f"""Ты решаешь бинарную классификацию товара.\n\n{rules}\n\nНазвание:\n{name}\n\nОписание:\n{desc}\n\nНа изображении объединены все фотографии товара.\n\nПредскажи целевую метку из обучающей разметки.\nОтветь строго одним символом: 0 или 1.\n\nОтвет:"""\n\ndef prepare_processor(model_path):\n    processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)\n    processor.tokenizer.padding_side = "left"\n    if processor.tokenizer.pad_token_id is None:\n        processor.tokenizer.pad_token = processor.tokenizer.eos_token\n    try:\n        processor.image_processor.size["longest_edge"] = 576 * 576\n        processor.image_processor.size["shortest_edge"] = 224 * 224\n    except Exception:\n        pass\n    z = processor.tokenizer.encode("0", add_special_tokens=False)\n    o = processor.tokenizer.encode("1", add_special_tokens=False)\n    if len(z) != 1 or len(o) != 1:\n        raise RuntimeError(f"Labels must be single tokens: 0={z}, 1={o}")\n    return processor, z[0], o[0]\n\ndef find_text_linear_modules(model):\n    targets = []\n    for name, module in model.named_modules():\n        if not name.startswith("model.language_model."):\n            continue\n        if "linear" not in module.__class__.__name__.lower():\n            continue\n        if name.endswith("lm_head"):\n            continue\n        targets.append(name)\n    targets = sorted(set(targets))\n    if len(targets) < 50:\n        raise RuntimeError(f"Too few LoRA targets: {len(targets)}")\n    return targets\n\ndef load_model(model_path, args):\n    qcfg = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.float16,\n    )\n    print("Loading Qwen3.5-4B NF4...", flush=True)\n    model = Qwen3_5ForConditionalGeneration.from_pretrained(\n        model_path,\n        quantization_config=qcfg,\n        device_map={"": 0},\n        attn_implementation="sdpa",\n        local_files_only=True,\n    )\n    model.tie_weights()\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    targets = find_text_linear_modules(model)\n    print("LoRA targets:", len(targets), flush=True)\n    lcfg = LoraConfig(\n        r=args.r,\n        lora_alpha=args.alpha,\n        lora_dropout=args.dropout,\n        bias="none",\n        target_modules=targets,\n        task_type="CAUSAL_LM",\n    )\n    model = get_peft_model(model, lcfg)\n    model.print_trainable_parameters()\n    return model, targets\n\ndef make_full_inputs(processor, row, label_ids, max_chars):\n    target = str(int(row["label"]))\n    messages = [\n        {"role":"user","content":[\n            {"type":"image"},\n            {"type":"text","text":build_prompt(row, max_chars)},\n        ]},\n        {"role":"assistant","content":[{"type":"text","text":target}]},\n    ]\n    try:\n        text = processor.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=False, enable_thinking=False\n        )\n    except TypeError:\n        text = processor.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=False\n        )\n    with Image.open(row["sheet_path"]) as im:\n        image = im.convert("RGB").copy()\n    inputs = processor(text=[text], images=[image], padding=False, return_tensors="pt")\n    target_id = label_ids[int(row["label"])]\n    ids = inputs["input_ids"][0]\n    pos = (ids == target_id).nonzero(as_tuple=False).flatten()\n    if len(pos) == 0:\n        raise RuntimeError(f"Target token not found for id={row[\'id\']}")\n    labels = torch.full_like(inputs["input_ids"], -100)\n    labels[0, int(pos[-1])] = target_id\n    inputs["labels"] = labels\n    return {k:v.to("cuda:0") for k,v in inputs.items() if isinstance(v, torch.Tensor)}\n\ndef make_prompt_inputs(processor, row, max_chars):\n    messages = [{"role":"user","content":[\n        {"type":"image"},\n        {"type":"text","text":build_prompt(row, max_chars)},\n    ]}]\n    try:\n        text = processor.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False\n        )\n    except TypeError:\n        text = processor.apply_chat_template(\n            messages, tokenize=False, add_generation_prompt=True\n        )\n    with Image.open(row["sheet_path"]) as im:\n        image = im.convert("RGB").copy()\n    inputs = processor(text=[text], images=[image], padding=False, return_tensors="pt")\n    return {k:v.to("cuda:0") for k,v in inputs.items() if isinstance(v, torch.Tensor)}\n\ndef balanced_indices(df, seed, enabled):\n    rng = np.random.default_rng(seed)\n    n = len(df)\n    if not enabled:\n        idx = np.arange(n)\n        rng.shuffle(idx)\n        return idx\n    counts = df["label"].value_counts().to_dict()\n    weights = np.array([1.0 / counts[int(y)] for y in df["label"]], dtype=np.float64)\n    weights /= weights.sum()\n    return rng.choice(np.arange(n), size=n, replace=True, p=weights)\n\ndef smoke(model, processor, train_df, label_ids, args):\n    print("=== QLORA SMOKE ===", flush=True)\n    state = {\n        n:p.detach().cpu().clone()\n        for n,p in model.named_parameters() if p.requires_grad\n    }\n    rows = []\n    for label in [0,1]:\n        part = train_df[train_df["label"] == label]\n        if len(part):\n            rows.append(part.iloc[0])\n    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr)\n    opt.zero_grad(set_to_none=True)\n    losses = []\n    for row in rows:\n        inputs = make_full_inputs(processor, row, label_ids, args.max_description_chars)\n        with torch.autocast("cuda", dtype=torch.float16):\n            out = model(**inputs, use_cache=False)\n            loss = out.loss\n        if not torch.isfinite(loss):\n            raise RuntimeError(f"Non-finite smoke loss {loss.item()}")\n        loss.backward()\n        losses.append(float(loss.detach().cpu()))\n    clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)\n    opt.step()\n    with torch.no_grad():\n        params = dict(model.named_parameters())\n        for n,v in state.items():\n            params[n].copy_(v.to(params[n].device, dtype=params[n].dtype))\n    del state, opt\n    gc.collect()\n    torch.cuda.empty_cache()\n    print("Smoke losses:", losses, flush=True)\n    print("QLORA SMOKE PASSED", flush=True)\n\ndef train(model, processor, train_df, label_ids, args, out_dir):\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    opt = torch.optim.AdamW(\n        trainable, lr=args.lr, weight_decay=args.weight_decay, betas=(0.9,0.999)\n    )\n    updates_per_epoch = math.ceil(len(train_df) / args.grad_accum)\n    total_updates = updates_per_epoch * args.epochs\n    warmup = max(1, int(total_updates * args.warmup_ratio))\n    sched = get_cosine_schedule_with_warmup(opt, warmup, total_updates)\n    opt.zero_grad(set_to_none=True)\n    logs = []\n    micro = 0\n    update = 0\n    t0 = time.time()\n\n    for epoch in range(args.epochs):\n        model.train()\n        idxs = balanced_indices(train_df, args.seed + epoch, bool(args.balanced_sampling))\n        recent = []\n        for j, idx in enumerate(idxs):\n            row = train_df.iloc[int(idx)]\n            inputs = make_full_inputs(processor, row, label_ids, args.max_description_chars)\n            with torch.autocast("cuda", dtype=torch.float16):\n                out = model(**inputs, use_cache=False)\n                raw_loss = out.loss\n                if not torch.isfinite(raw_loss):\n                    raise RuntimeError(f"Non-finite loss id={row[\'id\']}")\n                loss = raw_loss / args.grad_accum\n            loss.backward()\n            recent.append(float(raw_loss.detach().cpu()))\n            micro += 1\n\n            if micro % args.grad_accum == 0 or j == len(idxs)-1:\n                clip_grad_norm_(trainable, 1.0)\n                opt.step()\n                sched.step()\n                opt.zero_grad(set_to_none=True)\n                update += 1\n\n            if micro % args.log_every == 0 or j == len(idxs)-1:\n                rec = {\n                    "epoch":epoch+1,\n                    "micro_step":micro,\n                    "update_step":update,\n                    "loss":float(np.mean(recent[-args.log_every:])),\n                    "lr":float(opt.param_groups[0]["lr"]),\n                    "elapsed_min":(time.time()-t0)/60,\n                }\n                logs.append(rec)\n                print(\n                    f"[{args.category}] epoch {epoch+1}/{args.epochs} "\n                    f"{j+1}/{len(idxs)} loss={rec[\'loss\']:.4f} "\n                    f"lr={rec[\'lr\']:.2e} {rec[\'elapsed_min\']:.1f}m",\n                    flush=True,\n                )\n            del inputs, out, raw_loss, loss\n\n        pd.DataFrame(logs).to_csv(out_dir/"training_log.csv", index=False)\n\n@torch.inference_mode()\ndef evaluate(model, processor, val_df, zero_id, one_id, args):\n    model.eval()\n    rows = []\n    for i,(_,row) in enumerate(val_df.iterrows()):\n        inputs = make_prompt_inputs(processor, row, args.max_description_chars)\n        with torch.autocast("cuda", dtype=torch.float16):\n            out = model(**inputs, use_cache=False, logits_to_keep=1)\n        logits = out.logits[:, -1, :].float()\n        probs = torch.softmax(logits[:, [zero_id, one_id]], dim=-1)[0]\n        p0, p1 = float(probs[0].cpu()), float(probs[1].cpu())\n        rows.append({\n            "id":row["id"], "category":row["category"], "label":int(row["label"]),\n            "p0":p0, "p1":p1, "pred_05":int(p1 >= 0.5),\n        })\n        if (i+1) % 50 == 0:\n            print(f"[{args.category}] val {i+1}/{len(val_df)}", flush=True)\n        del inputs, out, logits, probs\n\n    pred = pd.DataFrame(rows)\n    f1_05 = float(f1_score(pred["label"], pred["pred_05"], zero_division=0))\n    acc = float(accuracy_score(pred["label"], pred["pred_05"]))\n    thresholds = np.linspace(0.05,0.95,181)\n    scores = [\n        float(f1_score(pred["label"], (pred["p1"] >= t).astype(int), zero_division=0))\n        for t in thresholds\n    ]\n    k = int(np.argmax(scores))\n    best_t, best_f1 = float(thresholds[k]), float(scores[k])\n    pred["pred_best"] = (pred["p1"] >= best_t).astype(int)\n    summary = {\n        "n_val":int(len(pred)),\n        "f1_at_0_5":f1_05,\n        "accuracy_at_0_5":acc,\n        "best_threshold":best_t,\n        "best_f1":best_f1,\n        "confusion_matrix_at_0_5":confusion_matrix(\n            pred["label"], pred["pred_05"], labels=[0,1]\n        ).tolist(),\n    }\n    return pred, summary\n\ndef main():\n    args = args_parser()\n    set_seed(args.seed)\n    out_dir = Path(args.output_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    df = pd.read_csv(args.manifest)\n    df = df[df["category"] == args.category].copy()\n    train_df = df[df["split"] == "train"].reset_index(drop=True)\n    val_df = df[df["split"] == "val"].reset_index(drop=True)\n\n    print("CATEGORY:", args.category, flush=True)\n    print("TRAIN:", len(train_df), "VAL:", len(val_df), flush=True)\n    print(train_df["label"].value_counts().sort_index(), flush=True)\n\n    processor, zero_id, one_id = prepare_processor(args.model_path)\n    model, targets = load_model(args.model_path, args)\n    label_ids = {0:zero_id, 1:one_id}\n\n    smoke(model, processor, train_df, label_ids, args)\n    if args.smoke_only:\n        print("SMOKE_ONLY DONE", flush=True)\n        return\n\n    train(model, processor, train_df, label_ids, args, out_dir)\n\n    # Saves adapter only, not the backbone.\n    model.save_pretrained(out_dir, safe_serialization=True)\n\n    pred, val_summary = evaluate(\n        model, processor, val_df, zero_id, one_id, args\n    )\n    pred.to_csv(out_dir/"val_predictions.csv", index=False)\n\n    summary = {\n        "base_model":"Qwen/Qwen3.5-4B",\n        "category":args.category,\n        "target":"direct data.csv label 0/1",\n        "train_rows":int(len(train_df)),\n        "val_rows":int(len(val_df)),\n        "lora":{\n            "r":args.r,\n            "alpha":args.alpha,\n            "dropout":args.dropout,\n            "target_module_count":len(targets),\n            "trainable_params":int(sum(\n                p.numel() for p in model.parameters() if p.requires_grad\n            )),\n        },\n        "training":{\n            "lr":args.lr,\n            "epochs":args.epochs,\n            "grad_accum":args.grad_accum,\n            "balanced_sampling":bool(args.balanced_sampling),\n            "max_description_chars":args.max_description_chars,\n        },\n        "validation":val_summary,\n    }\n    (out_dir/"target_modules.json").write_text(\n        json.dumps(targets, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    (out_dir/"training_summary.json").write_text(\n        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print("=== ADAPTER SAVED ===", out_dir, flush=True)\n    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print(TRAIN_SCRIPT, TRAIN_SCRIPT.stat().st_size, "bytes")

/kaggle/working/ecup_phase3/train_qwen_qlora.py 16557 bytes


## Smoke test

Один реальный QLoRA forward/backward на GPU0. Если здесь проблема с 4-bit, PEFT, processor или backward — полный train не стартует.

In [8]:
# 6.5 FIX Qwen3.5 fast kernels

!pip uninstall -y fla-core flash-linear-attention causal-conv1d >/dev/null 2>&1 || true

!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation

  Using cached flash_linear_attention-0.5.2-py3-none-any.whl.metadata (45 kB)
  Using cached fla_core-0.5.2-py3-none-any.whl.metadata (45 kB)
Using cached fla_core-0.5.2-py3-none-any.whl (819 kB)
Using cached flash_linear_attention-0.5.2-py3-none-any.whl (399 kB)
  Using cached causal_conv1d-1.6.2.post1.tar.gz (29 kB)
  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl size=193835395 sha256=c16c1c48d4fa63415cc797e02d69f97248c57c04627d99e394d5bb0ef266e288
  Stored in directory: /root/.cache/pip/wheels/ea/f0/26/5d87ae05a302e6dc8016c50cd8c7ee779585f593b9580e7cf8
Successfully built causal-conv1d


In [9]:
# 6.6 VERIFY kernels

import importlib.util
import importlib.metadata as md

assert importlib.util.find_spec("fla") is not None, "fla НЕ установлен"
assert importlib.util.find_spec("causal_conv1d") is not None, "causal_conv1d НЕ установлен"

print("flash-linear-attention:", md.version("flash-linear-attention"))
print("fla-core:", md.version("fla-core"))
print("causal-conv1d:", md.version("causal-conv1d"))

print("=== FAST KERNEL PACKAGES OK ===")

flash-linear-attention: 0.5.2
fla-core: 0.5.2
causal-conv1d: 1.6.2.post1
=== FAST KERNEL PACKAGES OK ===


In [10]:
# 6.7 MEMORY PATCH FOR T4

# Для первого нормального запуска уменьшаем adapter,
# но всё ещё оставляем LoRA на всех text-linear слоях.
LORA_R = 16
LORA_ALPHA = 32

# Картинки на диске НЕ пересоздаём.
# Processor сам уменьшит 576x576 -> максимум ~448x448.
TRAINER_TEXT = TRAIN_SCRIPT.read_text(encoding="utf-8")

# 1. Меньше visual tokens
TRAINER_TEXT = TRAINER_TEXT.replace(
    'processor.image_processor.size["longest_edge"] = 576 * 576',
    'processor.image_processor.size["longest_edge"] = 448 * 448'
)

# 2. Явно грузим НЕ-квантизованные части модели в FP16
TRAINER_TEXT = TRAINER_TEXT.replace(
    'quantization_config=qcfg,\n        device_map={"": 0},',
    'quantization_config=qcfg,\n        dtype=torch.float16,\n        device_map={"": 0},'
)

# 3. gradient checkpointing без reentrant input-grad hook
TRAINER_TEXT = TRAINER_TEXT.replace(
    'model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)',
    '''model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )'''
)

# 4. PEFT после prepare_model_for_kbit_training апкастит
# крупные frozen параметры в FP32. На T4 возвращаем frozen
# embeddings/vision/прочие ненормировочные параметры обратно в FP16.
needle = '''model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    targets = find_text_linear_modules(model)'''

replacement = '''model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    # Memory-critical для T4:
    # frozen non-quantized параметры держим FP16,
    # нормы оставляем FP32 для стабильности.
    for name, param in model.named_parameters():
        if (
            not param.requires_grad
            and param.dtype == torch.float32
            and param.__class__.__name__ != "Params4bit"
            and "norm" not in name.lower()
        ):
            param.data = param.data.to(torch.float16)

    model.tie_weights()
    torch.cuda.empty_cache()

    print(
        "GPU after kbit prep:",
        round(torch.cuda.memory_allocated() / 2**30, 2),
        "GB allocated"
    )

    targets = find_text_linear_modules(model)'''

assert needle in TRAINER_TEXT, "Не найден блок prepare_model_for_kbit_training"

TRAINER_TEXT = TRAINER_TEXT.replace(needle, replacement)

TRAIN_SCRIPT.write_text(
    TRAINER_TEXT,
    encoding="utf-8"
)

print("Patched:", TRAIN_SCRIPT)
print("LORA_R =", LORA_R)
print("LORA_ALPHA =", LORA_ALPHA)
print("max image processor area = 448x448")

Patched: /kaggle/working/ecup_phase3/train_qwen_qlora.py
LORA_R = 16
LORA_ALPHA = 32
max image processor area = 448x448


In [11]:
# 7. Smoke
SMOKE_DIR = WORK/"smoke"
shutil.rmtree(SMOKE_DIR, ignore_errors=True)

cmd = [
    "python","-u",str(TRAIN_SCRIPT),
    "--model_path",MODEL_PATH,
    "--manifest",str(MANIFEST),
    "--category","БАД",
    "--output_dir",str(SMOKE_DIR),
    "--r",str(LORA_R),
    "--alpha",str(LORA_ALPHA),
    "--dropout",str(LORA_DROPOUT),
    "--lr",str(LR),
    "--epochs","1",
    "--grad_accum","1",
    "--max_description_chars",str(MAX_DESCRIPTION_CHARS),
    "--seed",str(SEED),
    "--smoke_only",
]
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

rc = subprocess.run(cmd, env=env).returncode
assert rc == 0, f"Smoke failed: {rc}"
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
print("=== QLORA SMOKE PASSED ===")

CATEGORY: БАД
TRAIN: 7123 VAL: 346
label
0    1817
1    5306
Name: count, dtype: int64
Loading Qwen3.5-4B NF4...


Loading weights: 100%|██████████| 723/723 [00:09<00:00, 79.33it/s] 


GPU after kbit prep: 3.07 GB allocated
LoRA targets: 248
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101
=== QLORA SMOKE ===
Smoke losses: [0.7876251935958862, 0.48926645517349243]
QLORA SMOKE PASSED
SMOKE_ONLY DONE
=== QLORA SMOKE PASSED ===


## Full QLoRA — две категории параллельно

In [12]:
# 8. Train both adapters
BAD_DIR = ADAPTERS_DIR/"qwen35_4b_BAD_qlora"
FIRE_DIR = ADAPTERS_DIR/"qwen35_4b_FIRE_qlora"
shutil.rmtree(BAD_DIR, ignore_errors=True)
shutil.rmtree(FIRE_DIR, ignore_errors=True)

def make_cmd(category, out_dir):
    return [
        "python","-u",str(TRAIN_SCRIPT),
        "--model_path",MODEL_PATH,
        "--manifest",str(MANIFEST),
        "--category",category,
        "--output_dir",str(out_dir),
        "--r",str(LORA_R),
        "--alpha",str(LORA_ALPHA),
        "--dropout",str(LORA_DROPOUT),
        "--lr",str(LR),
        "--epochs",str(EPOCHS),
        "--grad_accum",str(GRAD_ACCUM),
        "--max_description_chars",str(MAX_DESCRIPTION_CHARS),
        "--balanced_sampling","1" if BALANCED_SAMPLING else "0",
        "--seed",str(SEED),
        "--log_every","25",
    ]

env0 = os.environ.copy()
env0["CUDA_VISIBLE_DEVICES"] = "0"
env0["TOKENIZERS_PARALLELISM"] = "false"

env1 = os.environ.copy()
env1["CUDA_VISIBLE_DEVICES"] = "1"
env1["TOKENIZERS_PARALLELISM"] = "false"

print("GPU0 -> БАД")
p0 = subprocess.Popen(make_cmd("БАД",BAD_DIR), env=env0)

print("GPU1 -> Легковоспламеняющиеся")
p1 = subprocess.Popen(make_cmd("Легковоспламеняющиеся",FIRE_DIR), env=env1)

rc0 = p0.wait()
rc1 = p1.wait()

print("BAD rc:",rc0,"FIRE rc:",rc1)
assert rc0 == 0, "BAD training failed"
assert rc1 == 0, "FIRE training failed"
print("=== BOTH TRAININGS FINISHED ===")

GPU0 -> БАД
GPU1 -> Легковоспламеняющиеся
CATEGORY: БАД
TRAIN: 7123 VAL: 346
label
0    1817
1    5306
Name: count, dtype: int64
CATEGORY: Легковоспламеняющиеся
TRAIN: 5248 VAL: 254
label
0    5059
1     189
Name: count, dtype: int64
Loading Qwen3.5-4B NF4...
Loading Qwen3.5-4B NF4...


Loading weights: 100%|██████████| 723/723 [00:13<00:00, 53.67it/s] 


GPU after kbit prep: 3.08 GB allocated
LoRA targets: 248
GPU after kbit prep: 3.08 GB allocated
LoRA targets: 248
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101
=== QLORA SMOKE ===
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101
=== QLORA SMOKE ===
Smoke losses: [0.1691066473722458, 0.3192494809627533]
QLORA SMOKE PASSED
Smoke losses: [0.7876251935958862, 0.48926645517349243]
QLORA SMOKE PASSED
[Легковоспламеняющиеся] epoch 1/1 25/5248 loss=0.6745 lr=9.38e-06 1.2m
[БАД] epoch 1/1 25/7123 loss=0.6842 lr=6.82e-06 1.5m
[Легковоспламеняющиеся] epoch 1/1 50/5248 loss=0.5528 lr=1.88e-05 2.5m
[БАД] epoch 1/1 50/7123 loss=0.4123 lr=1.36e-05 3.1m
[Легковоспламеняющиеся] epoch 1/1 75/5248 loss=0.4729 lr=2.81e-05 3.9m
[БАД] epoch 1/1 75/7123 loss=0.3187 lr=2.05e-05 4.5m
[Легковоспламеняющиеся] epoch 1/1 100/5248 loss=0.6830 lr=3.75e-05 5.3m
[БАД] epoch 1/1 100/7123 loss=0.6281 lr=2.73e-05 6.0m
[Легковоспламеняющиеся] epoch 1/1 125/5

In [13]:
# 9. Results
def load_json(path):
    with open(path,"r",encoding="utf-8") as f:
        return json.load(f)

bad = load_json(BAD_DIR/"training_summary.json")
fire = load_json(FIRE_DIR/"training_summary.json")

table = pd.DataFrame([
    {
        "category":"БАД",
        "train_rows":bad["train_rows"],
        "val_rows":bad["val_rows"],
        "r":bad["lora"]["r"],
        "trainable_params":bad["lora"]["trainable_params"],
        "f1@0.5":bad["validation"]["f1_at_0_5"],
        "best_threshold":bad["validation"]["best_threshold"],
        "best_f1":bad["validation"]["best_f1"],
    },
    {
        "category":"Легковоспламеняющиеся",
        "train_rows":fire["train_rows"],
        "val_rows":fire["val_rows"],
        "r":fire["lora"]["r"],
        "trainable_params":fire["lora"]["trainable_params"],
        "f1@0.5":fire["validation"]["f1_at_0_5"],
        "best_threshold":fire["validation"]["best_threshold"],
        "best_f1":fire["validation"]["best_f1"],
    },
])

display(table)
print("Mean F1 @0.5:", float(table["f1@0.5"].mean()))
print("Mean tuned F1:", float(table["best_f1"].mean()))
table.to_csv(WORK/"phase3_qwen_qlora_summary.csv",index=False)

,category,train_rows,val_rows,r,trainable_params,f1@0.5,best_threshold,best_f1
0,БАД,7123,346,16,32464896,0.938856,0.85,0.942346
1,Легковоспламеняющиеся,5248,254,16,32464896,0.823529,0.05,0.823529


Mean F1 @0.5: 0.8811927137718993
Mean tuned F1: 0.8829376681089931


In [14]:
# 10. Verify adapter files
for folder in [BAD_DIR,FIRE_DIR]:
    print("\n",folder.name)
    for p in sorted(folder.rglob("*")):
        if p.is_file():
            print(p.relative_to(folder), f"{p.stat().st_size/2**20:.2f} MB")
    assert (folder/"adapter_config.json").exists()
    assert list(folder.glob("adapter_model*.safetensors"))
print("\nAdapters verified.")


 qwen35_4b_BAD_qlora
README.md 0.01 MB
adapter_config.json 0.00 MB
adapter_model.safetensors 123.92 MB
target_modules.json 0.01 MB
training_log.csv 0.02 MB
training_summary.json 0.00 MB
val_predictions.csv 0.02 MB

 qwen35_4b_FIRE_qlora
README.md 0.01 MB
adapter_config.json 0.00 MB
adapter_model.safetensors 123.92 MB
target_modules.json 0.01 MB
training_log.csv 0.01 MB
training_summary.json 0.00 MB
val_predictions.csv 0.02 MB

Adapters verified.


In [15]:
# 11. ZIP adapters for download
BAD_ZIP = Path(shutil.make_archive(
    "/kaggle/working/qwen35_BAD_qlora_adapter",
    "zip",
    root_dir=str(BAD_DIR),
))
FIRE_ZIP = Path(shutil.make_archive(
    "/kaggle/working/qwen35_FIRE_qlora_adapter",
    "zip",
    root_dir=str(FIRE_DIR),
))

combined = Path("/kaggle/working/qwen35_phase3_adapters")
shutil.rmtree(combined, ignore_errors=True)
combined.mkdir(parents=True)

shutil.copytree(BAD_DIR, combined/"BAD")
shutil.copytree(FIRE_DIR, combined/"FIRE")
shutil.copy2(WORK/"validation_ids.csv", combined/"validation_ids.csv")
shutil.copy2(WORK/"phase3_qwen_qlora_summary.csv", combined/"phase3_qwen_qlora_summary.csv")

COMBINED_ZIP = Path(shutil.make_archive(
    "/kaggle/working/qwen35_phase3_adapters",
    "zip",
    root_dir=str(combined),
))

for p in [BAD_ZIP,FIRE_ZIP,COMBINED_ZIP]:
    print(p, f"{p.stat().st_size/2**20:.2f} MB")
    display(FileLink(str(p)))

print("СКАЧАЙ ZIP ДО ЗАВЕРШЕНИЯ KAGGLE SESSION.")

/kaggle/working/qwen35_BAD_qlora_adapter.zip 114.83 MB


/kaggle/working/qwen35_BAD_qlora_adapter.zip

/kaggle/working/qwen35_FIRE_qlora_adapter.zip 114.73 MB


/kaggle/working/qwen35_FIRE_qlora_adapter.zip

/kaggle/working/qwen35_phase3_adapters.zip 229.56 MB


/kaggle/working/qwen35_phase3_adapters.zip

СКАЧАЙ ZIP ДО ЗАВЕРШЕНИЯ KAGGLE SESSION.


In [16]:
# 12. Final saved summary
phase3 = {
    "base_model":MODEL_ID,
    "target":"direct data.csv label 0/1",
    "validation_n":int(len(val_df)),
    "contact_sheet_size":CONTACT_SHEET_SIZE,
    "lora_r":LORA_R,
    "lora_alpha":LORA_ALPHA,
    "lr":LR,
    "epochs":EPOCHS,
    "grad_accum":GRAD_ACCUM,
    "bad":bad,
    "fire":fire,
    "mean_f1_at_0_5":float(table["f1@0.5"].mean()),
    "mean_best_f1":float(table["best_f1"].mean()),
}
with open("/kaggle/working/phase3_result.json","w",encoding="utf-8") as f:
    json.dump(phase3,f,ensure_ascii=False,indent=2)
print(json.dumps(phase3,ensure_ascii=False,indent=2))

{
  "base_model": "Qwen/Qwen3.5-4B",
  "target": "direct data.csv label 0/1",
  "validation_n": 600,
  "contact_sheet_size": 576,
  "lora_r": 16,
  "lora_alpha": 32,
  "lr": 0.0001,
  "epochs": 1,
  "grad_accum": 8,
  "bad": {
    "base_model": "Qwen/Qwen3.5-4B",
    "category": "БАД",
    "target": "direct data.csv label 0/1",
    "train_rows": 7123,
    "val_rows": 346,
    "lora": {
      "r": 16,
      "alpha": 32,
      "dropout": 0.05,
      "target_module_count": 248,
      "trainable_params": 32464896
    },
    "training": {
      "lr": 0.0001,
      "epochs": 1,
      "grad_accum": 8,
      "balanced_sampling": true,
      "max_description_chars": 2200
    },
    "validation": {
      "n_val": 346,
      "f1_at_0_5": 0.9388560157790927,
      "accuracy_at_0_5": 0.9104046242774566,
      "best_threshold": 0.8499999999999999,
      "best_f1": 0.9423459244532804,
      "confusion_matrix_at_0_5": [
        [
          77,
          11
        ],
        [
          20,
          

In [17]:
# 13. OPTIONAL cleanup — только после скачивания ZIP
# shutil.rmtree(HF_ROOT, ignore_errors=True)
# shutil.rmtree(SHEETS_DIR, ignore_errors=True)
# disk_status("After cleanup")